### Projet : Prédiction de la qualité de l'air dans les villes marocaines
# Étape 1 : Collecte des données
 On va :
1. Se connecter à l'API OpenWeatherMap
2. Collecter les données météo (température, humidité, pression, vent)
3. Collecter les données qualité de l'air (AQI, PM2.5)
4. Sauvegarder tout dans un fichier CSV

## Cellule 1 — Importer les bibliothèques

In [7]:
import requests
import pandas as pd
import time
from datetime import datetime
import os

print('Bibliothèques importées avec succès !')

Bibliothèques importées avec succès !


## Cellule 2 — Mettre ta clé API et la liste des villes


In [8]:

API_KEY = "2a34926160a60e10a031804ae5f539e8"

# Les URLs de l'API OpenWeatherMap
URL_METEO = "https://api.openweathermap.org/data/2.5/weather"
URL_AIR   = "http://api.openweathermap.org/data/2.5/air_pollution"

VILLES = ["Casablanca", "Rabat", "Marrakech", "Fes"]

print("Villes choisies :", VILLES)
print("Clé API définie :", API_KEY[:6], "... (cachée pour sécurité)")

Villes choisies : ['Casablanca', 'Rabat', 'Marrakech', 'Fes']
Clé API définie : 2a3492 ... (cachée pour sécurité)


## Cellule 3 — Fonction pour collecter les données météo d'une ville

In [9]:
def collecter_meteo(ville):
    params = {
        "q"     : ville + ",MA",   
        "appid" : API_KEY,
        "units" : "metric",        
        "lang"  : "fr"
    }

    reponse = requests.get(URL_METEO, params=params)

    if reponse.status_code != 200:
        print(f"  ERREUR pour {ville} : code {reponse.status_code}")
        return None

    data = reponse.json()

    return {
        "ville"            : ville,
        "timestamp"        : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "latitude"         : data["coord"]["lat"],
        "longitude"        : data["coord"]["lon"],
        "temperature"      : data["main"]["temp"],
        "temp_ressentie"   : data["main"]["feels_like"],
        "temp_min"         : data["main"]["temp_min"],
        "temp_max"         : data["main"]["temp_max"],
        "humidite"         : data["main"]["humidity"],
        "pression"         : data["main"]["pressure"],
        "vent_vitesse"     : data["wind"]["speed"],
        "vent_direction"   : data["wind"].get("deg", 0),
        "nuages"           : data["clouds"]["all"],
        "description_meteo": data["weather"][0]["description"],
        "visibilite"       : data.get("visibility", 0)
    }

print("Fonction collecter_meteo() créée avec succès !")

Fonction collecter_meteo() créée avec succès !


## Cellule 4 — Fonction pour collecter la qualité de l'air d'une ville

In [10]:
def collecter_air(ville, latitude, longitude):

    params = {
        "lat"   : latitude,
        "lon"   : longitude,
        "appid" : API_KEY
    }

    reponse = requests.get(URL_AIR, params=params)

    if reponse.status_code != 200:
        print(f"  ERREUR qualité air pour {ville} : code {reponse.status_code}")
        return None

    data = reponse.json()
    composants = data["list"][0]["components"]

    aqi = data["list"][0]["main"]["aqi"]

    niveaux_aqi = {1: "Bon", 2: "Correct", 3: "Modéré", 4: "Mauvais", 5: "Très mauvais"}

    return {
        "aqi"          : aqi,
        "aqi_label"    : niveaux_aqi.get(aqi, "Inconnu"),
        "pm2_5"        : composants.get("pm2_5", 0),
        "pm10"         : composants.get("pm10", 0),
        "no2"          : composants.get("no2", 0),
        "o3"           : composants.get("o3", 0),
        "co"           : composants.get("co", 0),
        "so2"          : composants.get("so2", 0)
    }

print("Fonction collecter_air() créée avec succès !")

Fonction collecter_air() créée avec succès !


## Cellule 5 — Tester sur UNE seule ville (avant de tout lancer)

In [12]:
# Test rapide sur Fès
print("Test de connexion à l'API...")
print("="*40)

test_meteo = collecter_meteo("Fes")

if test_meteo:
    print(f"Ville          : {test_meteo['ville']}")
    print(f"Température    : {test_meteo['temperature']} °C")
    print(f"Humidité       : {test_meteo['humidite']} %")
    print(f"Pression       : {test_meteo['pression']} hPa")
    print(f"Vent           : {test_meteo['vent_vitesse']} m/s")
    print(f"Météo          : {test_meteo['description_meteo']}")
    print("")

    test_air = collecter_air("Fes", test_meteo["latitude"], test_meteo["longitude"])
    if test_air:
        print(f"AQI            : {test_air['aqi']} ({test_air['aqi_label']})")
        print(f"PM2.5          : {test_air['pm2_5']} µg/m³")
        print(f"PM10           : {test_air['pm10']} µg/m³")
        print("")
        print("SUCCÈS ! L'API fonctionne bien. Tu peux lancer la collecte complète.")
else:
    print("ERREUR ! Vérifie que ta clé API est bien écrite dans la Cellule 2.")

Test de connexion à l'API...
Ville          : Fes
Température    : 23.14 °C
Humidité       : 49 %
Pression       : 1008 hPa
Vent           : 3.09 m/s
Météo          : nuageux

AQI            : 2 (Correct)
PM2.5          : 4.99 µg/m³
PM10           : 10.49 µg/m³

SUCCÈS ! L'API fonctionne bien. Tu peux lancer la collecte complète.


## Cellule 6 — Collecter UNE fois pour les 4 villes et sauvegarder en CSV

In [13]:
def collecter_toutes_les_villes():

    lignes = []

    for ville in VILLES:
        print(f"Collecte pour {ville}...")

        # 1. Collecter la météo
        meteo = collecter_meteo(ville)
        if meteo is None:
            continue

        # 2. Collecter la qualité de l'air
        air = collecter_air(ville, meteo["latitude"], meteo["longitude"])
        if air is None:
            continue

        # 3. Fusionner météo + air en une seule ligne
        ligne = {**meteo, **air}
        lignes.append(ligne)

        print(f"  OK — Température: {meteo['temperature']}°C | PM2.5: {air['pm2_5']} | AQI: {air['aqi_label']}")

        # Pause pour ne pas surcharger l'API
        time.sleep(1)

    return lignes

FICHIER_CSV = "data_raw.csv"

print("Début de la collecte...")
print("="*40)
nouvelles_lignes = collecter_toutes_les_villes()

df_nouveau = pd.DataFrame(nouvelles_lignes)

if os.path.exists(FICHIER_CSV):
    df_existant = pd.read_csv(FICHIER_CSV)
    df_final = pd.concat([df_existant, df_nouveau], ignore_index=True)
else:
    df_final = df_nouveau

df_final.to_csv(FICHIER_CSV, index=False)

print("="*40)
print(f"SAUVEGARDÉ ! Le fichier '{FICHIER_CSV}' contient maintenant {len(df_final)} lignes.")
print("")
print("Aperçu des données collectées :")
df_nouveau

Début de la collecte...
Collecte pour Casablanca...
  OK — Température: 19.13°C | PM2.5: 1.17 | AQI: Modéré
Collecte pour Rabat...
  OK — Température: 19.04°C | PM2.5: 3.9 | AQI: Modéré
Collecte pour Marrakech...
  OK — Température: 22.04°C | PM2.5: 2.37 | AQI: Modéré
Collecte pour Fes...
  OK — Température: 23.14°C | PM2.5: 4.99 | AQI: Correct
SAUVEGARDÉ ! Le fichier 'data_raw.csv' contient maintenant 4 lignes.

Aperçu des données collectées :


,ville,timestamp,latitude,longitude,temperature,temp_ressentie,temp_min,temp_max,humidite,pression,...,description_meteo,visibilite,aqi,aqi_label,pm2_5,pm10,no2,o3,co,so2
0,Casablanca,2026-05-08 19:38:25,33.5928,-7.6192,19.13,18.74,19.07,19.19,63,1011,...,ciel dégagé,10000,3,Modéré,1.17,4.97,0.69,103.43,106.44,1.34
1,Rabat,2026-05-08 19:38:28,33.9911,-6.8401,19.04,18.64,19.04,19.04,63,1010,...,nuageux,8000,3,Modéré,3.90,8.93,2.13,104.25,105.58,4.22
2,Marrakech,2026-05-08 19:38:31,31.6315,-8.0083,22.04,21.50,22.04,22.04,46,1011,...,ciel dégagé,10000,3,Modéré,2.37,8.36,1.31,101.15,103.67,1.15
3,Fes,2026-05-08 19:38:34,34.0372,-4.9998,23.14,22.79,23.14,23.14,49,1008,...,nuageux,10000,2,Correct,4.99,10.49,0.95,90.74,98.58,0.75


## Cellule 7 — Collecter automatiquement toutes les heures pendant plusieurs jours

In [ ]:

NOMBRE_DE_COLLECTES = 72
INTERVALLE_SECONDES = 3600  # 3600 secondes = 1 heure

print(f"Démarrage de la collecte automatique")
print(f"Nombre de collectes prévues : {NOMBRE_DE_COLLECTES}")
print(f"Intervalle : toutes les {INTERVALLE_SECONDES // 3600} heure(s)")
print(f"Durée totale estimée : {NOMBRE_DE_COLLECTES} heures")
print("="*50)

for i in range(NOMBRE_DE_COLLECTES):
    maintenant = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"\nCollecte {i+1}/{NOMBRE_DE_COLLECTES} — {maintenant}")
    print("-"*40)

    # Collecter les données
    nouvelles_lignes = collecter_toutes_les_villes()

    # Sauvegarder dans le CSV
    df_nouveau = pd.DataFrame(nouvelles_lignes)

    if os.path.exists(FICHIER_CSV):
        df_existant = pd.read_csv(FICHIER_CSV)
        df_final = pd.concat([df_existant, df_nouveau], ignore_index=True)
    else:
        df_final = df_nouveau

    df_final.to_csv(FICHIER_CSV, index=False)

    total_lignes = len(df_final)
    print(f"CSV mis à jour : {total_lignes} lignes au total")

    if i + 1 == NOMBRE_DE_COLLECTES:
        print("\nCollecte terminée !")
        break

    print(f"Prochaine collecte dans 1 heure...")
    time.sleep(INTERVALLE_SECONDES)